# Clase 16: Implementación robusta de Dijkstra

## Pregunta inicial

Ya entendemos la idea de Dijkstra. ¿Cómo se convierte en una implementación confiable, reutilizable y testeable?

> [!IMPORTANT]
> Responde las preguntas y actividades en `notebook.md`, no dentro de este archivo.

**Responde esta pregunta en notebook.md.**


## Objetivos

Al terminar podrás:

- leer una implementación desde su contrato y no solo desde su bucle;
- justificar tipos, normalización y copia defensiva;
- distinguir TypeError, ValueError y KeyError;
- detectar entradas obsoletas y explicar el invariante del ciclo;
- separar cálculo, reconstrucción y coordinación;
- derivar pruebas desde contratos y riesgos;
- depurar y escribir revisiones accionables;
- conectar BFS, Dijkstra, Kruskal y topológico con su operación dominante.


## 1. De algoritmo correcto a software confiable

En la Clase 15 entendimos Dijkstra: distancias tentativas, relajación, min-heap y predecesores. Sin embargo, el pseudocódigo supone que el grafo está bien formado, que todos los pesos son válidos y que cada nodo existe en las tablas. Una función que solo traduce el bucle puede acertar en el ejemplo de clase y fallar con una entrada ligeramente distinta.

Hoy leeremos la implementación como un sistema de promesas. Debe responder qué acepta, qué devuelve, qué no modifica, cómo informa errores y qué invariante sostiene el ciclo. La robustez no reemplaza al algoritmo: crea las condiciones para que su razonamiento sea válido.

Ejemplo: `{"A": [("B", 2)]}` describe B aunque B no sea clave. Una implementación profesional decide explícitamente si lo acepta y normaliza, o si lo rechaza. La referencia lo normaliza sin cambiar el objeto recibido.

> [!IMPORTANT]
> “Produce el resultado esperado en un ejemplo” no equivale a “cumple un contrato reusable”.

Primero aprenderemos un orden de lectura que evita perdernos en los detalles.

### Comprueba tu comprensión

**Pregunta:** ¿Qué responsabilidades aparecen al pasar del pseudocódigo a una función reutilizable?

**Responde esta pregunta en notebook.md.**


## 2. Un orden profesional de lectura

Leer código no significa empezar en la primera línea del cuerpo y simular cada variable. Usaremos esta secuencia:

1. **Contrato:** firma, tipos, retorno y excepciones.
2. **Fronteras:** validaciones y normalización.
3. **Estado:** qué estructuras nacen antes del ciclo.
4. **Invariante:** qué debe ser cierto en cada iteración.
5. **Salida:** cómo representa éxito, inalcanzable y error.
6. **Pruebas:** qué ejemplo falsaría cada promesa.

La pregunta “¿qué intenta hacer esta línea?” es menos útil que “¿qué promesa protege y cómo lo demostraríamos?”. Por ejemplo, `if distancia_extraida != distancias[actual]: continue` protege el invariante de procesar solo candidaturas vigentes. La prueba correspondiente crea una ruta directa costosa que después mejora.

> [!TIP]
> Anota contrato → invariante → riesgo → prueba. Si no puedes proponer una prueba, probablemente aún no entendiste la decisión.

Aplicaremos el orden desde las declaraciones de tipos del módulo.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué conviene leer firma y docstring antes que el while principal?

**Responde esta pregunta en notebook.md.**


## 3. Tipos como decisiones de diseño

La referencia declara:

```python
Peso = int | float
GrafoPonderado = Mapping[str, Sequence[tuple[str, Peso]]]
```

`Mapping` expresa que necesitamos leer pares clave–valor, no que el objeto deba ser exactamente un `dict`. `Sequence` admite listas y tuplas. Esto amplía la interfaz sin debilitarla: aún exigimos nodos `str` y aristas de dos elementos.

Los type hints ayudan a personas, editores y analizadores, pero no validan entradas durante la ejecución. Por eso la función contiene comprobaciones explícitas. También debemos recordar una sutileza de Python: `bool` hereda de `int`. Si solo preguntamos `isinstance(peso, (int, float))`, `True` se aceptaría como peso 1 aunque no represente una medida del dominio.

> [!NOTE]
> Una anotación describe intención; una validación protege el contrato en tiempo de ejecución.

Ya sabemos qué se acepta conceptualmente; ahora veremos cómo convertirlo en una representación interna uniforme.

### Comprueba tu comprensión

**Pregunta:** ¿Qué diferencia práctica existe entre aceptar Mapping/Sequence y exigir dict/list?

**Responde esta pregunta en notebook.md.**


## 4. Normalización y copia defensiva

`_normalizar_grafo` tiene dos trabajos relacionados: validar la frontera y construir una copia interna predecible. Después de llamarla, el algoritmo puede confiar en que cada nodo es `str`, cada arista contiene vecino y peso, cada peso es `float` finito no negativo y todo vecino también aparece como clave.

La copia defensiva evita compartir listas con la entrada. Así, agregar un vecino implícito a `resultado` no modifica el diccionario original. Esta separación reduce el número de condiciones dentro del bucle de relajación.

```text
entrada flexible y no confiable
        ↓ validar + copiar
representación interna uniforme
        ↓
bucle de Dijkstra más pequeño
```

Una función auxiliar con guion bajo no es “menos importante”: encapsula una política interna. Se prueba directamente en el repositorio docente porque un fallo allí afecta todas las funciones públicas.

> [!CAUTION]
> Copiar solo el diccionario exterior no basta si todavía se comparten las listas de aristas.

Ahora clasificaremos los errores que la normalización debe comunicar.

### Comprueba tu comprensión

**Pregunta:** ¿Qué dos problemas resuelve _normalizar_grafo antes de ejecutar Dijkstra?

**Responde esta pregunta en notebook.md.**


## 5. TypeError y ValueError cuentan historias distintas

Usamos `TypeError` cuando el dato no tiene la clase de representación esperada: el grafo no es un mapeo, un nodo no es `str`, una arista no es un par o un peso no es numérico. Usamos `ValueError` cuando el tipo es admisible pero el valor viola el dominio: peso negativo, infinito o NaN.

| Entrada | Tipo | Problema | Excepción |
| --- | --- | --- | --- |
| `[("A", [])]` | lista | el grafo no es Mapping | TypeError |
| `{"A": [(2, 1)]}` | vecino int | vecino no es str | TypeError |
| `{"A": [("B", "3")]}` | peso str | no numérico | TypeError |
| `{"A": [("B", -1)]}` | número | dominio inválido | ValueError |
| `{"A": [("B", nan)]}` | número | no finito | ValueError |

Los mensajes no sustituyen al tipo de excepción, pero hacen accionable el fallo. Las pruebas pueden comprobar ambos sin acoplarse a cada carácter del texto.

Ya clasificamos los errores; examinemos dos casos que suelen pasar revisiones superficiales.

### Comprueba tu comprensión

**Pregunta:** ¿Cuándo corresponde TypeError y cuándo ValueError en esta implementación?

**Responde esta pregunta en notebook.md.**


## 6. Bool, NaN e infinito

`True` es instancia de `int` en Python. La condición robusta lo excluye antes de admitir enteros y flotantes. `NaN`, en cambio, sí es `float`, pero rompe la intuición de comparación: `nan < 3`, `nan > 3` y `nan == 3` son falsos. Un heap que contiene NaN deja de representar con claridad un orden de prioridades.

`math.isfinite` rechaza NaN, infinito positivo e infinito negativo. El infinito usado para distancias iniciales pertenece al estado interno; no es un peso válido de entrada. Esa distinción evita confundir “todavía no conozco una ruta” con “existe una arista infinita”.

```python
if isinstance(peso, bool) or not isinstance(peso, (int, float)):
    raise TypeError(...)
if not math.isfinite(peso):
    raise ValueError(...)
```

> [!WARNING]
> Que Python permita una operación no significa que el dominio del problema deba aceptarla.

Con pesos válidos, falta asegurar que las tablas incluyan todos los nodos descritos.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué True y NaN requieren comprobaciones específicas?

**Responde esta pregunta en notebook.md.**


## 7. Vecinos implícitos y representación total

En una lista de adyacencia dirigida es común que un destino sin aristas salientes aparezca solo dentro de una tupla. `{"A": [("B", 2)]}` sigue describiendo dos nodos. La normalización agrega B a la copia interna con lista vacía.

Esto garantiza que las comprensiones de distancias y predecesores produzcan tablas totales. También evita `KeyError` cuando B sea extraído y consultemos `normalizado[B]`. La entrada original permanece igual; completar la representación interna no equivale a corregirla en el lugar.

Contraejemplo: si inicializamos distancias únicamente con las claves originales, `distancias["B"]` no existe durante la primera relajación. Una prueba pequeña detecta el problema mejor que un grafo grande.

> [!TIP]
> Diseña la entrada mínima que alcance una línea riesgosa. Para este contrato bastan A y B.

La frontera ya está protegida. Entremos ahora en la función principal con premisas confiables.

### Comprueba tu comprensión

**Pregunta:** ¿Qué fallo evita resultado.setdefault(vecino, [])?

**Responde esta pregunta en notebook.md.**


## 8. El contrato de dijkstra

`dijkstra(grafo, origen)` resuelve caminos mínimos desde un origen a todos los nodos alcanzables. Devuelve un mapa de costos y un mapa de predecesores. Un camino particular se reconstruye después.

Esta separación conserva información y evita repetir el algoritmo para consultar destinos distintos. Los inalcanzables son nodos válidos con distancia `math.inf` y predecesor `None`; un origen inexistente es un error y produce `KeyError`.

| Situación | Representación |
| --- | --- |
| origen | distancia 0, predecesor None |
| alcanzable | costo finito, posible predecesor |
| inalcanzable | infinito, predecesor None |
| nodo inexistente solicitado | KeyError |

> [!IMPORTANT]
> Casos distintos merecen resultados distintos; de lo contrario el código llamador no puede tomar una decisión informada.

Con el retorno claro, construiremos un estado inicial que satisfaga el contrato para todos los nodos.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué dijkstra devuelve dos diccionarios en lugar de un camino?

**Responde esta pregunta en notebook.md.**


## 9. Estado inicial y tablas totales

Después de normalizar, creamos una entrada para cada nodo en ambas tablas. El origen cambia de infinito a cero y el heap nace con `(0.0, origen)`.

```python
distancias = {nodo: math.inf for nodo in normalizado}
predecesores = {nodo: None for nodo in normalizado}
distancias[origen] = 0.0
pendientes = [(0.0, origen)]
```

El invariante inicial es pequeño: cada clave interna existe en ambos mapas; `distancias[n]` es el menor costo descubierto hasta el momento; cada par del heap es una candidatura histórica. No afirmamos que cada par sea vigente: una mejora posterior puede dejar entradas viejas.

Un diseño “crear cuando haga falta” reduce unas líneas, pero aumenta condiciones y hace ambiguo si una ausencia significa nodo inválido, inalcanzable o todavía no visto.

Ya tenemos estado coherente; ahora veremos por qué extraer del heap no basta para decidir procesar.

### Comprueba tu comprensión

**Pregunta:** ¿Qué invariante establecen las comprensiones antes del while?

**Responde esta pregunta en notebook.md.**


## 10. El guard clause de entradas obsoletas

Cuando una distancia mejora, insertamos otra tupla y dejamos la anterior dentro del heap. Al extraer, comparamos:

```python
distancia_extraida, actual = heapq.heappop(pendientes)
if distancia_extraida != distancias[actual]:
    continue
```

Solo si la candidatura coincide con el valor vigente recorremos las aristas. Este guard clause convierte una situación excepcional en una salida temprana y mantiene el cuerpo principal sin un nivel extra de indentación.

Ejemplo: B entra con 20 y después mejora a 3. `(3, B)` procesa vecinos; cuando salga `(20, B)`, se descarta. No necesitamos borrar una posición arbitraria del heap.

> [!NOTE]
> La entrada obsoleta no es corrupción: es una consecuencia prevista de una implementación simple con `heapq`.

Con una candidatura vigente, el núcleo algorítmico vuelve a ser la relajación estudiada.

### Comprueba tu comprensión

**Pregunta:** ¿Qué garantiza la comparación inmediatamente posterior a heappop?

**Responde esta pregunta en notebook.md.**


## 11. Relajación y actualización atómica

El bucle calcula `candidata = distancia_extraida + peso`. Si es estrictamente menor, actualiza distancia y predecesor y agrega la nueva prioridad. Las tres acciones representan una sola decisión lógica.

```python
if candidata < distancias[vecino]:
    distancias[vecino] = candidata
    predecesores[vecino] = actual
    heapq.heappush(pendientes, (candidata, vecino))
```

Actualizar solo el costo puede hacer que todas las pruebas numéricas pasen y que la ruta sea falsa. Actualizar el predecesor sin insertar la candidatura impide propagar la mejora. Insertar sin cambiar la tabla crea una entrada que será obsoleta inmediatamente.

El `<` estricto conserva la primera ruta encontrada cuando hay empate. Otra política sería posible, pero debe documentarse y probarse.

Ya calculamos evidencia suficiente; ahora separaremos la reconstrucción del recorrido principal.

### Comprueba tu comprensión

**Pregunta:** ¿Qué datos deben actualizarse juntos cuando una candidata mejora?

**Responde esta pregunta en notebook.md.**


## 12. Reconstruir es otro problema

`reconstruir_camino` no necesita el grafo ni el heap. Sigue una cadena ya calculada. Si origen o destino no aparece en el mapa, el uso de la función es inválido y lanza `KeyError`. Si el destino sí existe pero su cadena llega a `None` sin alcanzar el origen, devuelve `[]`.

La lista se construye de destino a origen y se invierte solo al encontrar el origen. El conjunto `vistos` protege contra un mapa corrupto como A←B←A; sin él, la función podría no terminar.

Casos mínimos:

- origen=destino → `[origen]`;
- destino válido inalcanzable → `[]`;
- clave inexistente → `KeyError`;
- ciclo → `ValueError`.

> [!WARNING]
> No uses la misma salida para “no hay camino” y “el nodo ni siquiera existe”.

La función coordinadora combinará cálculo y reconstrucción sin duplicar lógica.

### Comprueba tu comprensión

**Pregunta:** ¿Qué diferencia hay entre destino inalcanzable y destino ausente?

**Responde esta pregunta en notebook.md.**


## 13. Coordinación sin duplicación

`camino_minimo` ejecuta Dijkstra una vez, verifica que el destino pertenezca al resultado y delega la secuencia a `reconstruir_camino`. No vuelve a implementar relajación ni recorre el grafo por su cuenta.

Este diseño crea una única fuente de verdad. Si cambiara la política de normalización, `camino_minimo` la recibiría automáticamente a través de `dijkstra`. También permite probar por separado cálculo, reconstrucción y coordinación.

```text
camino_minimo
├── dijkstra → distancias + predecesores
└── reconstruir_camino → secuencia
```

La función es pequeña, pero su prueba debe confirmar que costo y camino corresponden a la misma consulta, incluido `(math.inf, [])` para inalcanzables.

Ya entendimos las responsabilidades; convertiremos cada una en una matriz sistemática de pruebas.

### Comprueba tu comprensión

**Pregunta:** ¿Qué responsabilidades delega camino_minimo?

**Responde esta pregunta en notebook.md.**


## 14. Del contrato a una matriz de pruebas

Una suite profesional no colecciona ejemplos al azar. Cada prueba apunta a una promesa:

| Promesa | Entrada mínima | Afirmación |
| --- | --- | --- |
| corrección | ruta directa e indirecta | costo y camino |
| no mutación | lista guardada antes | igualdad después |
| vecino implícito | A→B sin clave B | B incluido |
| dominio | negativo/NaN/inf | ValueError |
| tipo | bool/str/None | TypeError |
| inalcanzable | dos componentes | inf y [] |
| eliminación perezosa | B=20, luego B=3 | ruta de costo 3 |
| defensa | ciclo en predecesores | ValueError |

`pytest.mark.parametrize` es útil cuando varias entradas prueban la misma regla. No debemos juntar contratos distintos en una parametrización que oculte qué falló.

Ahora aplicaremos esta matriz a una versión que parece razonable, pero es incompleta.

### Comprueba tu comprensión

**Pregunta:** ¿Qué dimensión del contrato no se verifica al probar únicamente el costo final?

**Responde esta pregunta en notebook.md.**


## 15. Auditoría de una implementación frágil

El archivo `src/dijkstra_para_revisar.py` contiene un bucle reconocible: heap, relajación y predecesores. Puede resolver el ejemplo conocido, pero no valida el grafo, acepta pesos negativos, no incluye vecinos implícitos, no rechaza origen ausente y procesa entradas obsoletas.

La actividad no consiste en decir “le falta validación”. Para cada hallazgo escribe:

1. contrato incumplido;
2. entrada mínima reproducible;
3. resultado observado;
4. resultado esperado;
5. prueba automatizada;
6. cambio localizado.

Ejemplo de comentario accionable: “Con `{"A": [("B", True)]}`, el peso se acepta como 1. El contrato admite números pero excluye bool. Agrega la comprobación antes de int/float y un test que espere TypeError”.

> [!CAUTION]
> No refactorices antes de reproducir: sin una prueba puedes cambiar comportamiento sin saber qué corregiste.

La auditoría nos prepara para depurar fallos con evidencia en vez de intuición.

### Comprueba tu comprensión

**Pregunta:** ¿Qué tres fallos reproducibles encuentras en dijkstra_para_revisar?

**Responde esta pregunta en notebook.md.**


In [ ]:
import ipywidgets as widgets
widgets.IntSlider(description="Prueba")


In [ ]:
from pathlib import Path
import sys
candidatas=[Path.cwd(),Path.cwd().parent,Path.cwd()/'clase_16',Path.cwd().parent/'clase_16']
RAIZ_CLASE=next((r for r in candidatas if (r/'src'/'visualizador_codigo.py').exists()),None)
if RAIZ_CLASE is None:
    raise FileNotFoundError('Abre desde curso-alumnos o clase_16/notebooks')
sys.path.insert(0,str(RAIZ_CLASE))
from src.visualizador_codigo import diagnosticar_widgets, mostrar_lectura_codigo
print(diagnosticar_widgets())
mostrar_lectura_codigo()


## 16. Clínica de depuración

Cuando pytest falla, lee primero el nombre del test y la última excepción propia, no toda la traza de arriba abajo. Reduce la entrada hasta conservar el fallo y decide qué frontera lo dejó pasar.

Flujo:

```text
fallo reproducible
→ contrato esperado
→ primera línea donde el estado deja de cumplirlo
→ prueba mínima
→ corrección localizada
→ suite completa
```

Un reporte útil incluye comando, entrada, esperado, observado y ubicación probable. “No funciona con grafos raros” no permite actuar. “`test_vecino_implicito` produce KeyError en `distancias[vecino]` porque B no se agregó al normalizar” sí.

Después de corregir, ejecuta el test mínimo y luego toda la suite para detectar regresiones. No borres una prueba solo porque descubre una decisión incómoda; revisa primero el contrato.

Con el fallo comprendido, la revisión de PR puede centrarse en comportamiento y evidencia.

### Comprueba tu comprensión

**Pregunta:** ¿Qué información mínima debe contener un reporte de fallo útil?

**Responde esta pregunta en notebook.md.**


## 17. Revisión profesional de código

Una revisión profesional separa comprensión, evidencia y recomendación. Primero resume el enfoque del autor. Después identifica una conducta observable, aporta una entrada o prueba y propone el cambio más pequeño que restaura el contrato.

Estructura:

```text
Contrato: los pesos deben ser finitos.
Evidencia: con math.nan no se lanza ValueError.
Impacto: el heap pierde una prioridad ordenable.
Acción: validar math.isfinite antes del signo.
Verificación: test parametrizado con nan e infinitos.
```

También se reconocen fortalezas específicas: “la normalización evita compartir listas” es más útil que “buen código”. El autor responde indicando qué aceptó, qué cambió y qué prueba lo demuestra.

> [!NOTE]
> Revisamos el comportamiento del código y la claridad de sus contratos, no a la persona.

La mantenibilidad también depende de entender dónde está realmente el costo algorítmico.

### Comprueba tu comprensión

**Pregunta:** ¿Qué hace que un comentario de revisión sea accionable y verificable?

**Responde esta pregunta en notebook.md.**


## 18. Complejidad sin perder robustez

Normalizar recorre nodos y aristas una vez: O(V+E) en tiempo y O(V+E) para la copia. Dijkstra con lista de adyacencia y heap permanece en O((V+E) log V). La fase lineal no domina al término logarítmico general.

Las validaciones locales son trabajo útil: evitan resultados sin garantía y permiten que el núcleo no repita comprobaciones. La defensa de ciclos en reconstrucción cuesta O(k), donde k es la longitud de la cadena consultada.

Optimizar eliminando validación antes de medir suele intercambiar claridad por un ahorro irrelevante. En cambio, sustituir el heap por buscar linealmente el mínimo sí cambia la operación dominante y puede llevar a O(V²).

> [!IMPORTANT]
> Robustez y eficiencia no son opuestos por defecto; deben analizarse con costos concretos.

Esta idea conecta Dijkstra con los otros algoritmos elegidos para el cierre.

### Comprueba tu comprensión

**Pregunta:** ¿La normalización cambia la complejidad asintótica de Dijkstra?

**Responde esta pregunta en notebook.md.**


## 19. Cuatro algoritmos, cuatro operaciones dominantes

El cierre del curso no será una lista de recetas. Compararemos decisiones:

| Algoritmo | Problema | Operación dominante | Estructura auxiliar |
| --- | --- | --- | --- |
| BFS | menor número de aristas | procesar por orden de descubrimiento | cola FIFO |
| Dijkstra | menor suma no negativa | extraer menor distancia tentativa | min-heap |
| Kruskal | árbol de expansión mínimo | unir componentes sin crear ciclo | ordenamiento + conjuntos disjuntos |
| Topológico | respetar dependencias | retirar nodos con grado de entrada cero | grados + cola |

La misma estructura no gana siempre. El heap de Dijkstra no reemplaza la cola FIFO de BFS cuando todos los costos son unitarios; conjuntos disjuntos resuelven conectividad entre componentes, no caminos desde un origen.

La pregunta permanente es: “¿qué operación hacemos una y otra vez y qué invariante debe conservarse?”.

Cerremos reconstruyendo la cadena completa desde contrato hasta prueba.

### Comprueba tu comprensión

**Pregunta:** ¿Qué estructura auxiliar se deriva de la operación dominante en cada algoritmo del cierre?

**Responde esta pregunta en notebook.md.**


## 20. Cierre

La implementación robusta comenzó antes del `while`: tipos amplios pero precisos, normalización, copia defensiva y errores diferenciados. Dentro del ciclo, el invariante se hizo visible con el descarte de obsoletas y la actualización conjunta de distancia, predecesor y heap. Después, reconstrucción y coordinación conservaron responsabilidades pequeñas.

| Pregunta | Respuesta de la clase |
| --- | --- |
| ¿Qué promete? | firma, docstring y excepciones |
| ¿Qué acepta? | representación validada |
| ¿Qué conserva? | tablas e invariante del heap |
| ¿Qué no modifica? | grafo recibido |
| ¿Cómo falla? | errores explícitos |
| ¿Cómo lo sabemos? | pruebas ligadas a contratos |
| ¿Cómo lo revisamos? | evidencia reproducible |

> [!IMPORTANT]
> Código profesional no es código largo: es código cuyas decisiones pueden explicarse, falsarse con pruebas y mantenerse sin duplicar responsabilidades.

Ticket de salida: escribe un contrato, el riesgo de romperlo y la prueba mínima que lo protege.

### Comprueba tu comprensión

**Pregunta:** ¿Qué cadena de lectura convierte una implementación en evidencia de confiabilidad?

**Responde esta pregunta en notebook.md.**
